# 🧠 Interactive Neural Network Digit Predictor (PyTorch Notebook Edition)

This notebook is for research purpose and is being used for benchmarks and evals in my current research paper "Beyond Attention and State" for the new neural achitecture proposal, please use the original one here https://github.com/S4nfs/NN-Digit-Predictor :

The UI is pure Python: a Gradio sketchpad sends each drawing to Python, PyTorch downsamples it to 28×28 and runs the forward pass, and matplotlib/Gradio components display the results. 

## 1. Setup

The next cell installs `torch`, `gradio` and `matplotlib` only if the active kernel doesn't already have them.

In [21]:
import importlib.util
import subprocess
import sys

REQUIREMENTS = {"torch": "torch", "gradio": "gradio>=5", "matplotlib": "matplotlib"}
missing = [spec for module, spec in REQUIREMENTS.items() if importlib.util.find_spec(module) is None]
if not missing:
    print("✅ All dependencies are already installed.")
elif importlib.util.find_spec("pip") is None:
    raise ModuleNotFoundError(
        f"Missing {', '.join(missing)} and pip is not available in this kernel's environment. "
        f"Install them with your package manager (e.g. `uv pip install {' '.join(missing)}`), then restart the kernel."
    )
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    importlib.invalidate_caches()
    print(f"Installed {', '.join(missing)}.")

✅ All dependencies are already installed.


In [22]:
import math
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display

# TF.js picked WebGL or CPU at runtime. This 784→128→64→10 MLP is so small that GPU launch overhead
# outweighs any speed-up, so the CPU is the fastest device here. Set "cuda" or "mps" if you prefer.
DEVICE = torch.device("cpu")

print(f"PyTorch {torch.__version__} · device: {DEVICE}")

PyTorch 2.11.0+cu128 · device: cpu


## 2. Model: port of `createModel()`

| Layer | TensorFlow.js | PyTorch |
|---|---|---|
| Input | `inputShape: [784]` (28×28 grid) | `[batch, 784]` float tensor |
| Hidden 1 | `dense(128, "relu", glorotNormal)` | `nn.Linear(784, 128)` + ReLU |
| Hidden 2 | `dense(64, "relu", glorotNormal)` | `nn.Linear(128, 64)` + ReLU |
| Output | `dense(10, "softmax", glorotNormal)` | `nn.Linear(64, 10)` + softmax |
| Optimizer | `tf.train.adam(0.01)` | `torch.optim.Adam(lr=0.01, eps=1e-7)` |
| Loss / metric | `categoricalCrossentropy` / `accuracy` | `nn.CrossEntropyLoss` on one-hot targets / argmax accuracy |

Three details keep the port faithful to TF.js:
- **Initialisation.** TF.js `glorotNormal` draws weights from a truncated normal with σ = √(2 / (fan_in + fan_out)), and biases start at zero. PyTorch's defaults differ, so the layers are re-initialised to match.
- **Softmax.** PyTorch merges the softmax into the loss for numerical stability, so `forward()` returns logits and `predict_proba()` applies the softmax. The result equals a softmax output layer followed by categorical cross-entropy.
- **Adam epsilon.** It is set to `1e-7`, the TF.js backend default.

In [23]:
INPUT_UNITS, HIDDEN_1_UNITS, HIDDEN_2_UNITS, OUTPUT_UNITS = 784, 128, 64, 10
LEARNING_RATE = 0.01
TFJS_EPSILON = 1e-7


def glorot_normal_(weight: torch.Tensor) -> torch.Tensor:
    """TF.js `glorotNormal`: truncated normal (cut at ±2σ) with σ = sqrt(2 / (fan_in + fan_out))."""
    fan_out, fan_in = weight.shape
    std = math.sqrt(2.0 / (fan_in + fan_out))
    return nn.init.trunc_normal_(weight, mean=0.0, std=std, a=-2.0 * std, b=2.0 * std)


class DigitNet(nn.Module):
    """Sequential MLP: 784 → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax)."""

    def __init__(self) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(INPUT_UNITS, HIDDEN_1_UNITS)     # Input (784) → Hidden 1 (128)
        self.hidden2 = nn.Linear(HIDDEN_1_UNITS, HIDDEN_2_UNITS)  # Hidden 1 → Hidden 2 (64)
        self.output = nn.Linear(HIDDEN_2_UNITS, OUTPUT_UNITS)     # Hidden 2 → Output (digits 0-9)
        for layer in (self.hidden1, self.hidden2, self.output):
            glorot_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.hidden1(x))
        x = F.relu(self.hidden2(x))
        return self.output(x)  # logits

    @torch.inference_mode()
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        return F.softmax(self(x), dim=-1)


def create_model() -> tuple[DigitNet, torch.optim.Optimizer]:
    """Port of `createModel()`: build the network and "compile" it with Adam(0.01)."""
    model = DigitNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999), eps=TFJS_EPSILON)
    return model, optimizer


# `loss: "categoricalCrossentropy"` (the softmax is fused into the loss; targets are one-hot vectors)
loss_fn = nn.CrossEntropyLoss()

summary_model = DigitNet()
print(summary_model)
print(f"Trainable parameters: {sum(p.numel() for p in summary_model.parameters()):,}")

DigitNet(
  (hidden1): Linear(in_features=784, out_features=128, bias=True)
  (hidden2): Linear(in_features=128, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=10, bias=True)
)
Trainable parameters: 109,386


## 3. Synthetic training data: port of `generateSyntheticData()`

The dataset is built exactly as in the original app:
- **10 digits × 15 variants = 150 samples** on a 28×28 grid. Each variant shifts the strokes by −1, 0 or +1 px horizontally and vertically.
- **Digital-style strokes.** Every digit is drawn from horizontal, vertical and diagonal lines (the stroke table below mirrors the `if/else` chain in `utils/nn.ts`).
- **Gaussian-like blur.** A 3×3 kernel (centre 0.4, edges 0.12, corners 0.03) smooths the strokes. Here it runs as a single `F.conv2d` call instead of nested loops.
- **Shuffle and labels.** A Fisher–Yates shuffle randomises the order, and the labels are one-hot encoded.

In [24]:
GRID_SIZE = 28
VARIANTS_PER_DIGIT = 15

# ("h", row, start_col, end_col) = drawHorizontalLine · ("v", col, start_row, end_row) = drawVerticalLine
# ("s", start_row, end_row, start_col, end_col) = drawSlash
DIGIT_STROKES = {
    0: [("h", 6, 8, 19), ("h", 21, 8, 19), ("v", 8, 6, 21), ("v", 19, 6, 21)],
    1: [("v", 14, 4, 23), ("s", 4, 8, 14, 10), ("h", 23, 10, 18)],
    2: [("h", 6, 8, 19), ("h", 14, 8, 19), ("h", 22, 8, 19), ("v", 19, 6, 14), ("v", 8, 14, 22)],
    3: [("h", 6, 8, 19), ("h", 14, 10, 19), ("h", 22, 8, 19), ("v", 19, 6, 22)],
    4: [("v", 18, 4, 23), ("v", 10, 4, 14), ("h", 14, 10, 22)],
    5: [("h", 6, 8, 19), ("h", 14, 8, 19), ("h", 22, 8, 19), ("v", 8, 6, 14), ("v", 19, 14, 22)],
    6: [("h", 6, 8, 19), ("h", 14, 8, 19), ("h", 22, 8, 19), ("v", 8, 6, 22), ("v", 19, 14, 22)],
    7: [("h", 6, 8, 19), ("s", 6, 22, 19, 10)],
    8: [("h", 6, 8, 19), ("h", 14, 8, 19), ("h", 22, 8, 19), ("v", 8, 6, 22), ("v", 19, 6, 22)],
    9: [("h", 6, 8, 19), ("h", 14, 8, 19), ("h", 22, 8, 19), ("v", 19, 6, 22), ("v", 8, 6, 14)],
}

# Same weights as the original blur loop: centre 0.4, edge neighbours 0.12, corner neighbours 0.03.
BLUR_KERNEL = torch.tensor(
    [[0.03, 0.12, 0.03],
     [0.12, 0.40, 0.12],
     [0.03, 0.12, 0.03]],
    dtype=torch.float64,
)


def draw_vertical_line(grid, col, start_row, end_row):
    grid[start_row:end_row + 1, col] = 1.0


def draw_horizontal_line(grid, row, start_col, end_col):
    grid[row, start_col:end_col + 1] = 1.0


def draw_slash(grid, start_row, end_row, start_col, end_col):
    steps = end_row - start_row
    for i in range(steps + 1):
        col = start_col + (i * (end_col - start_col)) / steps
        grid[start_row + i, math.floor(col + 0.5)] = 1.0  # JS Math.round (half up), not Python's half-to-even round()


def draw_digit(digit, shift_x, shift_y):
    grid = torch.zeros(GRID_SIZE, GRID_SIZE, dtype=torch.float64)
    for kind, *args in DIGIT_STROKES[digit]:
        if kind == "h":
            row, start_col, end_col = args
            draw_horizontal_line(grid, row + shift_y, start_col + shift_x, end_col + shift_x)
        elif kind == "v":
            col, start_row, end_row = args
            draw_vertical_line(grid, col + shift_x, start_row + shift_y, end_row + shift_y)
        else:
            start_row, end_row, start_col, end_col = args
            draw_slash(grid, start_row + shift_y, end_row + shift_y, start_col + shift_x, end_col + shift_x)
    return grid


def blur(grids):
    """Blur the interior pixels of a [N, 28, 28] stack; the 1-px border stays 0, as in the original loop."""
    blurred = torch.zeros_like(grids)
    blurred[:, 1:-1, 1:-1] = F.conv2d(grids.unsqueeze(1), BLUR_KERNEL.view(1, 1, 3, 3)).squeeze(1)
    return blurred.clamp(max=1.0)


def fisher_yates_shuffle(items):
    for i in range(len(items) - 1, 0, -1):
        j = random.randint(0, i)  # Math.floor(Math.random() * (i + 1))
        items[i], items[j] = items[j], items[i]
    return items


def generate_synthetic_data():
    """Port of `generateSyntheticData()`: returns xs [150, 784] in [0, 1] and one-hot ys [150, 10]."""
    grids, labels = [], []
    for digit in range(OUTPUT_UNITS):
        for variant in range(VARIANTS_PER_DIGIT):
            shift_x = (variant % 3) - 1  # shifts of -1, 0, 1
            shift_y = (variant // 3) % 3 - 1
            grids.append(draw_digit(digit, shift_x, shift_y))
            labels.append(digit)

    images = blur(torch.stack(grids)).reshape(-1, GRID_SIZE * GRID_SIZE)
    order = fisher_yates_shuffle(list(range(len(labels))))
    xs = images[order].to(torch.float32)
    ys = F.one_hot(torch.tensor(labels)[order], num_classes=OUTPUT_UNITS).to(torch.float32)
    return xs.to(DEVICE), ys.to(DEVICE)


def digit_gallery(xs, ys):
    """Inline preview of one training sample per digit (white strokes on black, like the drawing board)."""
    figures = []
    for digit in range(OUTPUT_UNITS):
        image = xs[ys.argmax(dim=1) == digit][0].reshape(GRID_SIZE, GRID_SIZE).tolist()
        pixels = "".join(
            f'<rect x="{c}" y="{r}" width="1" height="1" fill-opacity="{value:.3f}"/>'
            for r, row in enumerate(image) for c, value in enumerate(row) if value > 0
        )
        figures.append(
            '<figure style="margin:0;text-align:center;font:12px monospace;color:#64748b">'
            '<svg viewBox="0 0 28 28" width="56" height="56" style="display:block;background:#000;border-radius:6px">'
            f'<g fill="#fff" shape-rendering="crispEdges">{pixels}</g></svg>{digit}</figure>'
        )
    return HTML(f'<div style="display:flex;flex-wrap:wrap;gap:10px">{"".join(figures)}</div>')


xs, ys = generate_synthetic_data()
print(f"xs: {tuple(xs.shape)} {xs.dtype} · ys: {tuple(ys.shape)} one-hot · samples per digit: {ys.sum(dim=0).int().tolist()}")
display(digit_gallery(xs, ys))

xs: (150, 784) torch.float32 · ys: (150, 10) one-hot · samples per digit: [15, 15, 15, 15, 15, 15, 15, 15, 15, 15]


## 4. Live training: port of `trainModelLive()`

This follows `model.fit(xs, ys, { epochs: 45, batchSize: 16 })` in TF.js:
- Mini-batches of 16, with the data reshuffled every epoch (`shuffle: true` is the TF.js default).
- After each epoch, `on_epoch(epoch, loss, acc)` receives the sample-weighted mean loss and accuracy. In the app, this callback drives the live training screen.

The cell below trains a throwaway model as a quick headless check. The app trains its own model in section 8.

In [25]:
EPOCHS = 45
BATCH_SIZE = 16


def train_model_live(model, optimizer, on_epoch=None, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """Port of `trainModelLive()`: fit a fresh synthetic dataset, reporting (epoch, loss, acc) after every epoch."""
    xs, ys = generate_synthetic_data()
    n = xs.shape[0]
    model.train()
    for epoch in range(epochs):
        order = torch.randperm(n, device=xs.device)
        loss_sum, correct = 0.0, 0
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            logits = model(xs[batch])
            loss = loss_fn(logits, ys[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
            correct += (logits.argmax(dim=1) == ys[batch].argmax(dim=1)).sum().item()
        if on_epoch is not None:
            on_epoch(epoch + 1, loss_sum / n, correct / n)
    model.eval()
    # No `dispose()` needed: PyTorch frees xs/ys as soon as they go out of scope.


demo_model, demo_optimizer = create_model()
history = []
started = time.perf_counter()
train_model_live(demo_model, demo_optimizer, on_epoch=lambda epoch, loss, acc: history.append((epoch, loss, acc)))
elapsed = time.perf_counter() - started
for epoch, loss, acc in history[::11]:
    print(f"epoch {epoch:2d}/{EPOCHS} · loss {loss:.4f} · accuracy {acc:.1%}")
print(f"Trained {EPOCHS} epochs in {elapsed:.2f}s")

epoch  1/45 · loss 2.1069 · accuracy 24.0%
epoch 12/45 · loss 0.0008 · accuracy 100.0%
epoch 23/45 · loss 0.0003 · accuracy 100.0%
epoch 34/45 · loss 0.0002 · accuracy 100.0%
epoch 45/45 · loss 0.0001 · accuracy 100.0%
Trained 45 epochs in 0.93s


## 5. Inference: port of `handleDrawingUpdate()`

The browser sends the board as 784 floats in [0, 1], using the same 280→28 downsampling and luminance step as `DrawingCanvas.tsx`. The function below then follows the original:
- **Empty board.** All zeros returns all-zero probabilities and no prediction.
- **Forward pass.** It runs under `torch.inference_mode()`, PyTorch's counterpart to `tf.tidy()`. No autograd graph is recorded, and intermediate tensors are freed right away.
- **Threshold.** A digit is reported only when the top probability is above **15%**.

In [26]:
PREDICTION_THRESHOLD = 0.15


def predict_digit(model, pixel_matrix):
    """Returns (probabilities[10], predicted digit or None, forward-pass time in ms or None for an empty board)."""
    if all(value == 0 for value in pixel_matrix):
        return [0.0] * OUTPUT_UNITS, None, None
    started = time.perf_counter()
    x = torch.tensor(pixel_matrix, dtype=torch.float32, device=DEVICE).reshape(1, INPUT_UNITS)
    probabilities = model.predict_proba(x)[0].tolist()
    elapsed_ms = (time.perf_counter() - started) * 1000
    best = max(probabilities)
    digit = probabilities.index(best) if best > PREDICTION_THRESHOLD else None
    return probabilities, digit, elapsed_ms


for digit in range(OUTPUT_UNITS):
    sample = xs[ys.argmax(dim=1) == digit][0]
    probabilities, predicted, elapsed_ms = predict_digit(demo_model, sample.tolist())
    print(f"digit {digit} → predicted {predicted} ({max(probabilities):.1%}) in {elapsed_ms:.3f} ms")
print("empty board →", predict_digit(demo_model, [0.0] * INPUT_UNITS))

digit 0 → predicted 0 (100.0%) in 0.404 ms
digit 1 → predicted 1 (100.0%) in 0.294 ms
digit 2 → predicted 2 (100.0%) in 0.181 ms
digit 3 → predicted 3 (100.0%) in 0.171 ms
digit 4 → predicted 4 (100.0%) in 0.171 ms
digit 5 → predicted 5 (100.0%) in 0.157 ms
digit 6 → predicted 6 (100.0%) in 1.920 ms
digit 7 → predicted 7 (100.0%) in 0.274 ms
digit 8 → predicted 8 (100.0%) in 0.150 ms
digit 9 → predicted 9 (100.0%) in 0.158 ms
empty board → ([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], None, None)


## 6. User interface (pure Python)
The UI is built with [Gradio](https://www.gradio.app) using Python only:

In [27]:
import base64
import io
import queue
import threading

import gradio as gr
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

BLUE, SLATE_200, SLATE_300, SLATE_400, SLATE_500, SLATE_800 = "#2563eb", "#e2e8f0", "#cbd5e1", "#94a3b8", "#64748b", "#1e293b"
CANVAS_SIZE = 280
BRUSH_SIZE = 18

# gr.Sketchpad renders through PixiJS/WebGL, which is unavailable in VS Code's notebook output iframe (and in
# browsers with hardware acceleration off), so the board would not accept strokes. This plain 2D canvas works
# everywhere: it sends a PNG data URL to a hidden textbox on every stroke, and Python does all the processing.
BOARD_HTML = f"""<canvas id="digit-board" width="{CANVAS_SIZE}" height="{CANVAS_SIZE}"
    style="width:{CANVAS_SIZE}px;height:{CANVAS_SIZE}px;background:#fff;border:2px solid #cbd5e1;border-radius:12px;
    touch-action:none;cursor:crosshair;display:block;margin:auto"></canvas>"""
BOARD_JS = f"""() => {{
  const setup = () => {{
    const canvas = document.getElementById("digit-board");
    const box = document.querySelector("#board-data textarea");
    if (!canvas || !box) return setTimeout(setup, 100);
    if (canvas.dataset.ready) return;
    canvas.dataset.ready = "1";
    const ctx = canvas.getContext("2d");
    ctx.lineCap = ctx.lineJoin = "round";
    ctx.lineWidth = {BRUSH_SIZE};
    ctx.strokeStyle = "#000";
    let drawing = false, last = 0;
    const send = (force) => {{
      const now = Date.now();
      if (!force && now - last < 80) return;
      last = now;
      box.value = canvas.toDataURL("image/png");
      box.dispatchEvent(new Event("input", {{ bubbles: true }}));
    }};
    const pos = (e) => {{
      const r = canvas.getBoundingClientRect();
      return [(e.clientX - r.left) * canvas.width / r.width, (e.clientY - r.top) * canvas.height / r.height];
    }};
    canvas.addEventListener("pointerdown", (e) => {{
      drawing = true; canvas.setPointerCapture(e.pointerId);
      const [x, y] = pos(e); ctx.beginPath(); ctx.moveTo(x, y); ctx.lineTo(x, y); ctx.stroke(); send(false);
    }});
    canvas.addEventListener("pointermove", (e) => {{
      if (!drawing) return;
      const [x, y] = pos(e); ctx.lineTo(x, y); ctx.stroke(); send(false);
    }});
    const stop = () => {{ if (drawing) {{ drawing = false; send(true); }} }};
    canvas.addEventListener("pointerup", stop);
    canvas.addEventListener("pointercancel", stop);
    window.clearDigitBoard = () => ctx.clearRect(0, 0, canvas.width, canvas.height);
  }};
  setup();
}}"""
CLEAR_JS = "() => { window.clearDigitBoard && window.clearDigitBoard(); }"


def sketch_to_pixels(sketch):
    """Port of DrawingCanvas `triggerInference`: 280x280 board → 28x28 stroke intensity in [0, 1] (done in PyTorch)."""
    image = sketch.get("composite") if isinstance(sketch, dict) else sketch
    if isinstance(image, str):  # PNG data URL from the drawing board
        image = np.array(Image.open(io.BytesIO(base64.b64decode(image.split(",", 1)[-1]))).convert("RGBA")) if image else None
    if image is None:
        return [0.0] * INPUT_UNITS
    image = torch.as_tensor(image, dtype=torch.float32)
    # Ink is drawn in black on a white board; invert so strokes become 1.0 on a 0.0 background, like the
    # white-on-black canvas of the original. Transparent pixels (alpha 0) count as empty board.
    ink = 1.0 - image[..., :3].mean(dim=-1) / 255.0
    if image.shape[-1] == 4:
        ink = ink * image[..., 3] / 255.0
    small = F.interpolate(ink[None, None], size=(GRID_SIZE, GRID_SIZE), mode="area")
    return small.flatten().tolist()


def confidence_bars_figure(probabilities):
    """Port of ConfidenceBars.tsx."""
    best = max(probabilities)
    winner = probabilities.index(best) if best > PREDICTION_THRESHOLD else -1
    fig, ax = plt.subplots(figsize=(4.2, 3.6), dpi=100)
    digits = list(range(OUTPUT_UNITS))[::-1]
    values = [probabilities[d] * 100 for d in digits]
    ax.barh(digits, [100] * OUTPUT_UNITS, color="#f1f5f9", edgecolor=SLATE_200, height=0.6)
    ax.barh(digits, values, color=[BLUE if d == winner else SLATE_200 for d in digits], height=0.6)
    for d, v in zip(digits, values):
        ax.text(103, d, f"{v:.1f}%", va="center", fontsize=9, family="monospace",
                color=BLUE if d == winner else SLATE_500, fontweight="bold" if d == winner else "normal")
    ax.set_yticks(digits, [str(d) for d in digits])
    for label, d in zip(ax.get_yticklabels(), digits):
        label.set_color(BLUE if d == winner else SLATE_400)
        label.set_fontweight("bold")
    ax.set_xlim(0, 125)
    ax.set_xticks([])
    ax.tick_params(left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title("Model Probability Distribution", loc="left", fontsize=12, fontweight="bold", color=SLATE_800)
    fig.tight_layout()
    plt.close(fig)
    return fig


def network_figure(probabilities):
    """Port of NetworkVisualizer.tsx (same layout coordinates as the SVG)."""
    cols = [40, 150, 260, 380]
    labels = ["Input (784)", "Dense (128)", "Dense (64)", "Output (10)"]
    input_y = [40, 80, 120, 160, 200, 240]
    hidden1_y = [60, 100, 140, 180, 220]
    hidden2_y = [80, 120, 160, 200]
    output_y = [45, 65, 85, 105, 125, 145, 165, 185, 205, 225]
    best = max(probabilities)
    has_inference = best > PREDICTION_THRESHOLD
    winner = probabilities.index(best) if has_inference else -1

    fig, ax = plt.subplots(figsize=(5.2, 3.5), dpi=100)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#f8fafc")
    for (xa, ya_list), (xb, yb_list) in [((cols[0], input_y), (cols[1], hidden1_y)), ((cols[1], hidden1_y), (cols[2], hidden2_y))]:
        for ya in ya_list:
            for yb in yb_list:
                ax.plot([xa, xb], [ya, yb], color=SLATE_300, lw=0.5, alpha=0.4, zorder=1)
    for ya in hidden2_y:
        for idx, yb in enumerate(output_y):
            active = idx == winner
            ax.plot([cols[2], cols[3]], [ya, yb], color=BLUE if active else SLATE_300,
                    lw=1.5 if active else 0.5, ls="--" if active else "-", alpha=0.9 if active else 0.4, zorder=2 if active else 1)
    ax.scatter([cols[0]] * len(input_y), input_y, s=30, color=SLATE_500, zorder=3)
    for x, ys in ((cols[1], hidden1_y), (cols[2], hidden2_y)):
        ax.scatter([x] * len(ys), ys, s=30, color="#f1f5f9", edgecolors=SLATE_300, linewidths=1.5, zorder=3)
    for x in cols[:3]:
        ax.text(x, 260, "...", ha="center", fontsize=9, color=SLATE_400, fontweight="bold")
    for idx, y in enumerate(output_y):
        active = idx == winner
        alpha = max(0.1, probabilities[idx]) if has_inference else 1.0
        ax.scatter(cols[3], y, s=110 if active else 50, color=BLUE if active else "#f1f5f9",
                   edgecolors=BLUE if active else SLATE_300, linewidths=1.2, alpha=alpha, zorder=4)
        ax.text(cols[3] + 16, y, str(idx), va="center", fontsize=8, family="monospace", fontweight="bold",
                color=BLUE if active else SLATE_500)
    for x, label in zip(cols, labels):
        ax.text(x, 18, label, ha="center", fontsize=7.5, fontweight="semibold", color=SLATE_500)
    ax.set_xlim(0, 420)
    ax.set_ylim(280, 0)
    ax.axis("off")
    ax.set_title("Neural Network Architecture", loc="left", fontsize=12, fontweight="bold", color=SLATE_800)
    fig.tight_layout()
    plt.close(fig)
    return fig


def training_card(epoch, loss, accuracy, done=False):
    """Port of the training screen: progress bar plus Epoch / Loss / Accuracy."""
    percent = epoch / EPOCHS * 100
    title = "Model ready: draw a digit!" if done else "Training Neural Network"
    return f"""
<div style="border:1px solid {SLATE_200};border-radius:12px;padding:14px 18px;background:#fff">
  <div style="font-weight:700;color:{SLATE_800}">{'✅' if done else '⏳'} {title}</div>
  <div style="font-size:12px;color:{SLATE_500}">Compiling dense layers and fitting network weights to synthetic strokes...</div>
  <div style="margin:10px 0;height:8px;background:#f1f5f9;border:1px solid {SLATE_200};border-radius:9999px;overflow:hidden">
    <div style="width:{percent:.0f}%;height:100%;background:{BLUE}"></div></div>
  <div style="display:flex;justify-content:space-around;font-family:monospace;color:#334155">
    <span>EPOCH <b>{epoch}/{EPOCHS}</b></span><span>LOSS <b>{loss:.4f}</b></span><span>ACCURACY <b>{accuracy * 100:.1f}%</b></span>
  </div>
</div>"""


def prediction_card(digit, inference_ms):
    latency = "—" if inference_ms is None else f"~{inference_ms:.2f}ms"
    return (f"<div style='display:flex;align-items:center;justify-content:space-between;border:1px solid {SLATE_200};"
            f"border-radius:12px;padding:12px 18px;background:#fff'>"
            f"<div><div style='font-size:10px;color:{SLATE_400};font-weight:600;letter-spacing:.05em'>PREDICTION</div>"
            f"<div style='font-weight:700;color:{SLATE_800}'>Classified Digit</div>"
            f"<div style='font-size:12px;color:{SLATE_400}'>⚡ Inference: {latency}</div></div>"
            f"<div style='font-size:40px;font-weight:900;font-family:monospace;color:{BLUE}'>{'—' if digit is None else digit}</div></div>")

## 7. `DigitPredictorApp` owns the PyTorch model and optimizer, streams live training progress (on launch and on **Retrain Weights**, which fine-tunes the current weights like the original), ignores drawings while training, and turns each drawing into a prediction.

In [28]:
class DigitPredictorApp:
    def __init__(self, epoch_delay: float = 0.02):
        """`epoch_delay` is cosmetic: it lets the progress card animate (PyTorch trains in well under a second)."""
        self.model, self.optimizer = create_model()
        self.epoch_delay = epoch_delay
        self.is_training = False
        self.inference_ms = None
        self.demo = self._build()

    def train_stream(self):
        """Generator used by Gradio: yields the training card after every epoch."""
        if self.is_training:
            return
        self.is_training = True
        updates = queue.Queue()

        def on_epoch(epoch, loss, acc):
            updates.put((epoch, loss, acc))
            if self.epoch_delay:
                time.sleep(self.epoch_delay)

        def worker():
            try:
                train_model_live(self.model, self.optimizer, on_epoch=on_epoch)
            except Exception as err:
                print(f"Training error: {err!r}", file=sys.stderr)
            finally:
                updates.put(None)

        threading.Thread(target=worker, daemon=True).start()
        last = (0, 0.0, 0.0)
        yield training_card(*last)
        while (item := updates.get()) is not None:
            last = item
            yield training_card(*last)
        self.is_training = False
        yield training_card(*last, done=True)

    def reset_predictor(self):
        """Like React re-mounting the predictor after training: blank board, zero probabilities."""
        return "", *self._prediction_outputs([0.0] * OUTPUT_UNITS, None)

    def predict(self, sketch):
        """Port of `handleDrawingUpdate`."""
        if self.is_training:
            return gr.update(), gr.update(), gr.update()
        probabilities, digit, elapsed_ms = predict_digit(self.model, sketch_to_pixels(sketch))
        if elapsed_ms is not None:
            self.inference_ms = elapsed_ms if self.inference_ms is None else 0.8 * self.inference_ms + 0.2 * elapsed_ms
        return self._prediction_outputs(probabilities, digit)

    def _prediction_outputs(self, probabilities, digit):
        return prediction_card(digit, self.inference_ms), confidence_bars_figure(probabilities), network_figure(probabilities)

    def _build(self):
        css = "#board-data { display: none !important; }"
        with gr.Blocks(title="Handwritten Digit Predictor", css=css, js=BOARD_JS) as demo:
            gr.HTML(f"""<div style="text-align:center">
                <span style="font-size:12px;font-weight:600;color:#475569;background:#f1f5f9;border:1px solid {SLATE_200};
                border-radius:9999px;padding:4px 14px">🧠 PYTORCH NEURAL NETWORK</span>
                <h1 style="font-size:40px;font-weight:900;margin:12px 0 4px;color:#0f172a">Handwritten Digit Predictor</h1>
                <p style="color:{SLATE_500}">Draw a number from <b style="color:{BLUE}">0 to 9</b> on the board.
                The model will perform real-time forward propagation and predict it below!</p></div>""")
            status = gr.HTML(training_card(0, 0.0, 0.0))
            with gr.Row():
                with gr.Column(scale=1, min_width=300):
                    gr.Markdown("**Drawing board**: draw a digit with your mouse, pen or finger")
                    gr.HTML(BOARD_HTML)
                    board = gr.Textbox(elem_id="board-data", show_label=False, container=False)
                    clear = gr.Button("🧹 Clear Board")
                    prediction = gr.HTML(prediction_card(None, None))
                    retrain = gr.Button("🔄 Retrain Weights")
                with gr.Column(scale=1, min_width=300):
                    bars = gr.Plot(show_label=False)
                with gr.Column(scale=1, min_width=300):
                    network = gr.Plot(show_label=False)
            gr.Markdown("<center><small>© 2026 Interactive Neural Network Digit Predictor. Engineered with PyTorch & Gradio.</small></center>")

            predictor = [board, prediction, bars, network]
            demo.load(self.train_stream, outputs=status).then(self.reset_predictor, outputs=predictor).then(None, js=CLEAR_JS)  # train on "page mount"
            retrain.click(self.train_stream, outputs=status).then(self.reset_predictor, outputs=predictor).then(None, js=CLEAR_JS)
            clear.click(self.reset_predictor, outputs=predictor, js=CLEAR_JS)
            board.input(self.predict, inputs=board, outputs=[prediction, bars, network], trigger_mode="always_last", show_progress="hidden")
        return demo

## 8. Launch the app 🚀

The next cell starts the app inline in the notebook and trains the network live, like the original did on page mount. When the status card says **Model ready**, draw a digit on the white board; use **Clear Board** to start over and **Retrain Weights** to fine-tune again. If the inline view is too small, open the printed local URL in your browser.

> **Tip:** the network learns from the digital-style strokes shown in section 3 (box-shaped 0, 2, 3, 5, 6, 8, 9, a straight 1 with a base, an angular 4 and 7). Digits drawn in that style, centred and large, are recognised most reliably.

In [29]:
app = DigitPredictorApp()
app.demo.launch(inline=True, height=900, prevent_thread_lock=True, quiet=True)
print(f"App running at {app.demo.local_url}")

App running at http://127.0.0.1:7863/
